<a href="https://colab.research.google.com/github/cduffy57/Github-and-Jupyter-setup/blob/main/Final_Project_Checkpoint_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Final Project Checkpoint 3 — Q1 & Q2 Cleaning

**Team:** Omar Tall and Christopher Duffy  

**AI-use note:** This notebook was developed with assistance from ChatGPT (OpenAI) for code structure, variable checking, and debugging (Q1 and Q2). We reviewed and ran the code ourselves and are responsible for the cleaning decisions.

**Question 1:** Is previous flu vaccination history associated with participants' willingness to receive a flu vaccine?

**Question 2:** Are participants with lower income, less education, or no health insurance more likely to report lower vaccine intent?

In [ ]:
from google.colab import drive
import os
import pandas as pd
import numpy as np

drive.mount('/content/drive')

data_folder = "/content/drive/My Drive/Colab Notebooks/replication_package/2input_data/csv/"
main_survey_path = data_folder + "main_survey.csv"

if not os.path.exists(main_survey_path):
    raise FileNotFoundError(
        "main_survey.csv was not found. Run the class DTA-to-CSV conversion notebook first."
    )

raw_df = pd.read_csv(main_survey_path)

# The class conversion notebook saves the pandas index into the CSV.
# It is not part of the survey data, so remove it if it appears.
if "Unnamed: 0" in raw_df.columns:
    raw_df = raw_df.drop(columns="Unnamed: 0")

print("Raw shape:", raw_df.shape)
display(raw_df.head())
raw_df.info()


Mounted at /content/drive
Raw shape: (2893, 164)


,id,authority,video_length,video_race,el_status,el_progress,el_durationinseconds,el_finished,el_responseid,el_flushot,...,area_division,region,region_num,region_midwest,region_northeast,region_south,region_west,missing_region,resp_no_med,resp_any_med
0,18197,doc,long,white,NaN,NaN,NaN,NaN,R_6LsRVVTs2vjptgl,NaN,...,West South Central,South,South,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1,5465,doc,short,white,NaN,NaN,NaN,NaN,R_2P0vAZKzXespjdQ,NaN,...,West South Central,South,South,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2,7534,doc,short,white,NaN,NaN,NaN,NaN,R_10ryZI0l77UTzwy,NaN,...,South Atlantic,South,South,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,5456,lay,short,black,NaN,NaN,NaN,NaN,R_2tL0vrhh2xYKLKL,NaN,...,West North Central,Midwest,Midwest,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,4758,doc,long,white,NaN,NaN,NaN,NaN,R_2v8J0lMeEAma2rt,NaN,...,Middle Atlantic,Northeast,Northeast,0.0,1.0,0.0,0.0,0.0,1.0,0.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2893 entries, 0 to 2892
Columns: 164 entries, id to resp_any_med
dtypes: float64(67), int64(69), object(28)
memory usage: 3.6+ MB


## Keep and inspect the variables for Q1 and Q2

I kept the participant ID, flu-vaccine history, pre/post vaccine intent, income, education, insurance, and treatment group. raw_df stays unchanged.

I am also looking at the raw response counts before changing anything, especially income because the coding is a little unusual.

In [ ]:
q1_q2_df = raw_df[
    [
        "id",
        "last_flushot",
        "likelyshot_pre",
        "likelyshot_post",
        "income",
        "educ",
        "health_insurance",
        "treatment_group"
    ]
].copy()

print("Subset shape:", q1_q2_df.shape)

for col in ["last_flushot", "income", "educ", "health_insurance"]:
    print(f"\n{col}")
    print(q1_q2_df[col].value_counts(dropna=False).sort_index())

Subset shape: (2893, 8)

last_flushot
last_flushot
1    168
2    635
3    438
4    859
6    793
Name: count, dtype: int64

income
income
1     543
2     426
3     488
4     371
5     284
6     225
7     179
8     165
9      55
10     22
11    100
12     35
Name: count, dtype: int64

educ
educ
1     338
2    2555
Name: count, dtype: int64

health_insurance
health_insurance
1    1763
2    1046
3      84
Name: count, dtype: int64


## Clean the variables

For Question 1, last_flushot gives more detail than has_flushot, so I use it to create a simpler vaccination-history category.

For Question 2, I follow the coding used by the replication package:
- raw income code **11 = refused to answer**, so it becomes missing
- raw income code **12 = $150,000+**, so it becomes the final top income category
- education has only two groups because the study already focused on non-college participants
- insurance code **3** is treated as unsure/missing instead of forcing it into Yes or No


In [ ]:
# Q1: vaccination history and intent
q1_q2_df["resp_last_flushot"] = q1_q2_df["last_flushot"].replace(6, 0)

q1_q2_df["flushot_vacc_exp"] = q1_q2_df["resp_last_flushot"].map({
    0: "Never Taker",
    1: "Recent Taker",
    2: "Recent Taker",
    3: "Ever Taker",
    4: "Ever Taker"
})

q1_q2_df = q1_q2_df.rename(columns={
    "likelyshot_pre": "intent_pre",
    "likelyshot_post": "intent"
})

# Q2: income
q1_q2_df["resp_income"] = q1_q2_df["income"].astype("Int64")
q1_q2_df.loc[q1_q2_df["resp_income"] == 11, "resp_income"] = pd.NA
q1_q2_df.loc[q1_q2_df["resp_income"] == 12, "resp_income"] = 11

income_labels = {
    1: "$0-9,999",
    2: "$10,000-19,999",
    3: "$20,000-29,999",
    4: "$30,000-39,999",
    5: "$40,000-49,999",
    6: "$50,000-59,999",
    7: "$60,000-74,999",
    8: "$75,000-99,999",
    9: "$100,000-124,999",
    10: "$125,000-149,999",
    11: "$150,000+"
}
q1_q2_df["income_group"] = q1_q2_df["resp_income"].map(income_labels)

# Q2: education
q1_q2_df["resp_educ"] = q1_q2_df["educ"].astype("Int64")
q1_q2_df["resp_hs"] = (q1_q2_df["resp_educ"] == 2).astype("Int64")

# Q2: insurance
q1_q2_df["resp_hi"] = q1_q2_df["health_insurance"].astype("Int64")
q1_q2_df["resp_hi"] = q1_q2_df["resp_hi"].replace({
    1: 1,
    2: 0,
    3: pd.NA
}).astype("Int64")

final_columns = [
    "id",
    "treatment_group",
    "resp_last_flushot",
    "flushot_vacc_exp",
    "intent_pre",
    "intent",
    "resp_income",
    "income_group",
    "resp_educ",
    "resp_hs",
    "resp_hi"
]

clean_q1_q2 = q1_q2_df[final_columns].copy()

print("Vaccination history:")
print(clean_q1_q2["flushot_vacc_exp"].value_counts(dropna=False))

print("\nMissing values:")
print(clean_q1_q2.isna().sum())

print("\nExact duplicate rows:", clean_q1_q2.duplicated().sum())
print("Duplicate respondent IDs:", clean_q1_q2["id"].duplicated().sum())

assert clean_q1_q2["resp_last_flushot"].dropna().isin([0, 1, 2, 3, 4]).all()
assert clean_q1_q2["intent_pre"].dropna().between(0, 10).all()
assert clean_q1_q2["intent"].dropna().between(0, 10).all()
assert clean_q1_q2["resp_income"].dropna().between(1, 11).all()
assert clean_q1_q2["resp_educ"].dropna().isin([1, 2]).all()
assert clean_q1_q2["resp_hi"].dropna().isin([0, 1]).all()

print("\nValidation checks passed.")
print("Final shape:", clean_q1_q2.shape)
display(clean_q1_q2.head())


Vaccination history:
flushot_vacc_exp
Ever Taker      1297
Recent Taker     803
Never Taker      793
Name: count, dtype: int64

Missing values:
id                     0
treatment_group        0
resp_last_flushot      0
flushot_vacc_exp       0
intent_pre             0
intent                 0
resp_income          100
income_group         100
resp_educ              0
resp_hs                0
resp_hi               84
dtype: int64

Exact duplicate rows: 0
Duplicate respondent IDs: 0

Validation checks passed.
Final shape: (2893, 11)


,id,treatment_group,resp_last_flushot,flushot_vacc_exp,intent_pre,intent,resp_income,income_group,resp_educ,resp_hs,resp_hi
0,18197,black white doc long,2,Recent Taker,0,1,4,"$30,000-39,999",2,1,0
1,5465,black white doc short,0,Never Taker,0,0,6,"$50,000-59,999",2,1,0
2,7534,black white doc short,0,Never Taker,0,0,5,"$40,000-49,999",2,1,1
3,5456,black black lay short,3,Ever Taker,2,1,5,"$40,000-49,999",2,1,1
4,4758,black white doc long,0,Never Taker,2,4,1,"$0-9,999",2,1,1


## Cleaning decisions

- Used the converted main_survey.csv produced by the class conversion notebook.
- Kept the original imported DataFrame separate and made a smaller copy for Q1/Q2.
- Used last_flushot to represent prior flu-vaccination history.
- Renamed the pre/post willingness variables to intent_pre and intent.
- Treated refused income responses as missing and kept the top income bracket correctly.
- Kept the limited education categories because that is part of the study's sample design.
- Treated the unsure insurance response as missing rather than calling it insured or uninsured.
- Checked missing values and duplicate respondents but did not remove or fill responses just to make the data complete.
- Full analysis of the research questions comes later.

In [ ]:
output_file = "/content/drive/My Drive/Colab Notebooks/q1_q2_cleaned.csv"
clean_q1_q2.to_csv(output_file, index=False)

print("Saved cleaned file to:")
print(output_file)

Saved cleaned file to:
/content/drive/My Drive/Colab Notebooks/q1_q2_cleaned.csv


In [ ]:
import os
import pandas as pd


from google.colab import drive

drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# root_dir is the path to your My_Drive folder.
root_dir = "/content/drive/My Drive/"

# i'm storing the data folder in the Colab Notebooks directory at the top level of the Google Drive
data_folder = root_dir + "Colab Notebooks/replication_package/2input_data/"

# change directory to the data folder
os.chdir(data_folder)

In [ ]:
def dta_to_csv(dta_file_path):
  """
  takes a path to a .dta file and converts it to a .csv. The result
  is stored a subdirectory of the original directory named "/csv".

  Arguments:
    dta_file_path: Full path to the .dta file

  """

  # get the directory and file name from the full path
  directory_name, file_name = os.path.split(dta_file_path)

  # read the .dta file into a pandas DataFrame
  print("Converting file", dta_file_path)
  data = pd.io.stata.read_stata(dta_file_path)

  # save the DataFrame as a ".csv" to the "csv" directory in the original path (directory_name).
  # And get rid of the ".dta" by slicing the file_name
  data.to_csv(directory_name + "/csv/" +  file_name[:-4] +  ".csv")

In [ ]:
def convert_all_dta_to_csv(data_folder):
  """
  Traverse the directory that has path name data_folder. Look for
  .dta files and call dta_to_csv() to convert them to .csv
  """

  # Walk the directory structure
  for root, direc, files in os.walk(data_folder):
    for file in files:
      file_path = os.path.join(root, file)

      # split the file into its name and extension
      filename, extension = os.path.splitext(file)

      # does the file have a .dta extension
      if extension.lower() == ('.dta'):
        dir_path = root # store current directory

        # Create a 'csv' subdirectory if it doesn't exist
        csv_dir_path = os.path.join(dir_path, 'csv')
        if not os.path.exists(csv_dir_path):
          os.makedirs(csv_dir_path)

        # call the conversion function
        dta_to_csv(file_path)

# call the function to do the conversion
convert_all_dta_to_csv(data_folder)

Converting file /content/drive/My Drive/Colab Notebooks/replication_package/2input_data/brfss_2017.dta
Converting file /content/drive/My Drive/Colab Notebooks/replication_package/2input_data/main_survey.dta
Converting file /content/drive/My Drive/Colab Notebooks/replication_package/2input_data/main_survey_attrit_info.dta
Converting file /content/drive/My Drive/Colab Notebooks/replication_package/2input_data/redemption_data.dta
Converting file /content/drive/My Drive/Colab Notebooks/replication_package/2input_data/sender_rating_survey.dta


# Question 3

In [ ]:
data_folder = "/content/drive/My Drive/Colab Notebooks/replication_package/2input_data/csv/"
attrition_path = data_folder + "main_survey_attrit_info.csv"

if not os.path.exists(attrition_path):
    raise FileNotFoundError(
        "main_survey_attrit_info.csv was not found. Run the class DTA-to-CSV conversion notebook first."
    )

raw_attrit_df = pd.read_csv(attrition_path)

# The class conversion notebook saves the pandas index into the CSV.
# It is not part of the survey data, so remove it if it appears.
if "Unnamed: 0" in raw_attrit_df.columns:
    raw_attrit_df = raw_attrit_df.drop(columns="Unnamed: 0")

print("Raw shape:", raw_attrit_df.shape)
display(raw_attrit_df.head())
raw_attrit_df.info()


Raw shape: (3245, 10)


,pharmacy_lookup,sample,covid_vacc_trial,resp_black,flu_season,survey_platform,T,T_cc,T_lay,T_ackn
0,NaN,season 2 lucid2,NaN,1.0,1.0,Lucid,"D,AN",NaN,NaN,1.0
1,NaN,season 2 lucid,NaN,1.0,1.0,Lucid,D,0.0,NaN,0.0
2,NaN,season 2 lucid2,NaN,0.0,1.0,Lucid,CC,1.0,NaN,NaN
3,NaN,season 2 cloudresearch,NaN,1.0,1.0,CloudResearch/Facebook,"D,AN",NaN,NaN,1.0
4,NaN,season 2 lucid2,NaN,1.0,1.0,Lucid,"D,AN",NaN,NaN,1.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3245 entries, 0 to 3244
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   pharmacy_lookup   2914 non-null   float64
 1   sample            3245 non-null   object 
 2   covid_vacc_trial  2039 non-null   float64
 3   resp_black        3245 non-null   float64
 4   flu_season        3245 non-null   float64
 5   survey_platform   3245 non-null   object 
 6   T                 3245 non-null   object 
 7   T_cc              2264 non-null   float64
 8   T_lay             975 non-null    float64
 9   T_ackn            963 non-null    float64
dtypes: float64(7), object(3)
memory usage: 253.6+ KB


# Narrow down necessary data and Change column names

Go through each column to make this more readable.

In [ ]:
q3_df = raw_attrit_df[
    [ 'resp_black', 'T_cc', 'T_lay', 'T_ackn']
].copy()

q3_df = q3_df.rename(columns={
    'resp_black': 'Respondent Race',
    'T_cc': 'Race Concordance',
    'T_lay': 'Messenger Appearance',
    'T_ackn': 'Acknowledgement'
})

print("Subset shape:", q3_df.shape)

for col in ["Respondent Race", "Race Concordance", "Messenger Appearance", "Acknowledgement"]:
    print(f"\n{col}")
    print(q3_df[col].value_counts(dropna=False).sort_index())

Subset shape: (3245, 4)

Respondent Race
Respondent Race
0.0    1307
1.0    1938
Name: count, dtype: int64

Race Concordance
Race Concordance
0.0    1136
1.0    1128
NaN     981
Name: count, dtype: int64

Messenger Appearance
Messenger Appearance
0.0     479
1.0     496
NaN    2270
Name: count, dtype: int64

Acknowledgement
Acknowledgement
0.0     478
1.0     485
NaN    2282
Name: count, dtype: int64


Now go through the variables to name the ones with just 0 or 1 as responses.

In [ ]:
clean_q3_df = q3_df.replace({
    "Respondent Race":      {0: "Not Black", 1: "Black"},
    "Race Concordance":     {0: "Different Race", 1: "Same Race"},
    "Messenger Appearance": {0: "Expert", 1: "Layman"},
    "Acknowledgement":      {0: "No", 1: "Yes"},
})

In [ ]:
output_file = "/content/drive/My Drive/Colab Notebooks/q3_cleaned.csv"
clean_q3_df.to_csv(output_file, index=False)

print("Saved cleaned file to:")
print(output_file)

Saved cleaned file to:
/content/drive/My Drive/Colab Notebooks/q3_cleaned.csv


In [ ]:
import os
import pandas as pd


from google.colab import drive

drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# root_dir is the path to your My_Drive folder.
root_dir = "/content/drive/My Drive/"

# i'm storing the data folder in the Colab Notebooks directory at the top level of the Google Drive
data_folder = root_dir + "Colab Notebooks/replication_package/2input_data/"

# change directory to the data folder
os.chdir(data_folder)

In [ ]:
def dta_to_csv(dta_file_path):
  """
  takes a path to a .dta file and converts it to a .csv. The result
  is stored a subdirectory of the original directory named "/csv".

  Arguments:
    dta_file_path: Full path to the .dta file

  """

  # get the directory and file name from the full path
  directory_name, file_name = os.path.split(dta_file_path)

  # read the .dta file into a pandas DataFrame
  print("Converting file", dta_file_path)
  data = pd.io.stata.read_stata(dta_file_path)

  # save the DataFrame as a ".csv" to the "csv" directory in the original path (directory_name).
  # And get rid of the ".dta" by slicing the file_name
  data.to_csv(directory_name + "/csv/" +  file_name[:-4] +  ".csv")

In [ ]:
def convert_all_dta_to_csv(data_folder):
  """
  Traverse the directory that has path name data_folder. Look for
  .dta files and call dta_to_csv() to convert them to .csv
  """

  # Walk the directory structure
  for root, direc, files in os.walk(data_folder):
    for file in files:
      file_path = os.path.join(root, file)

      # split the file into its name and extension
      filename, extension = os.path.splitext(file)

      # does the file have a .dta extension
      if extension.lower() == ('.dta'):
        dir_path = root # store current directory

        # Create a 'csv' subdirectory if it doesn't exist
        csv_dir_path = os.path.join(dir_path, 'csv')
        if not os.path.exists(csv_dir_path):
          os.makedirs(csv_dir_path)

        # call the conversion function
        dta_to_csv(file_path)

# call the function to do the conversion
convert_all_dta_to_csv(data_folder)

Converting file /content/drive/My Drive/Colab Notebooks/replication_package/2input_data/brfss_2017.dta
Converting file /content/drive/My Drive/Colab Notebooks/replication_package/2input_data/main_survey.dta
Converting file /content/drive/My Drive/Colab Notebooks/replication_package/2input_data/main_survey_attrit_info.dta
Converting file /content/drive/My Drive/Colab Notebooks/replication_package/2input_data/redemption_data.dta
Converting file /content/drive/My Drive/Colab Notebooks/replication_package/2input_data/sender_rating_survey.dta


# Question 3

In [ ]:
data_folder = "/content/drive/My Drive/Colab Notebooks/replication_package/2input_data/csv/"
attrition_path = data_folder + "main_survey_attrit_info.csv"

if not os.path.exists(attrition_path):
    raise FileNotFoundError(
        "main_survey_attrit_info.csv was not found. Run the class DTA-to-CSV conversion notebook first."
    )

raw_attrit_df = pd.read_csv(attrition_path)

# The class conversion notebook saves the pandas index into the CSV.
# It is not part of the survey data, so remove it if it appears.
if "Unnamed: 0" in raw_attrit_df.columns:
    raw_attrit_df = raw_attrit_df.drop(columns="Unnamed: 0")

print("Raw shape:", raw_attrit_df.shape)
display(raw_attrit_df.head())
raw_attrit_df.info()


Raw shape: (3245, 10)


,pharmacy_lookup,sample,covid_vacc_trial,resp_black,flu_season,survey_platform,T,T_cc,T_lay,T_ackn
0,NaN,season 2 lucid2,NaN,1.0,1.0,Lucid,"D,AN",NaN,NaN,1.0
1,NaN,season 2 lucid,NaN,1.0,1.0,Lucid,D,0.0,NaN,0.0
2,NaN,season 2 lucid2,NaN,0.0,1.0,Lucid,CC,1.0,NaN,NaN
3,NaN,season 2 cloudresearch,NaN,1.0,1.0,CloudResearch/Facebook,"D,AN",NaN,NaN,1.0
4,NaN,season 2 lucid2,NaN,1.0,1.0,Lucid,"D,AN",NaN,NaN,1.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3245 entries, 0 to 3244
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   pharmacy_lookup   2914 non-null   float64
 1   sample            3245 non-null   object 
 2   covid_vacc_trial  2039 non-null   float64
 3   resp_black        3245 non-null   float64
 4   flu_season        3245 non-null   float64
 5   survey_platform   3245 non-null   object 
 6   T                 3245 non-null   object 
 7   T_cc              2264 non-null   float64
 8   T_lay             975 non-null    float64
 9   T_ackn            963 non-null    float64
dtypes: float64(7), object(3)
memory usage: 253.6+ KB


# Narrow down necessary data and Change column names

Go through each column to make this more readable.

In [ ]:
q3_df = raw_attrit_df[
    [ 'resp_black', 'T_cc', 'T_lay', 'T_ackn']
].copy()

q3_df = q3_df.rename(columns={
    'resp_black': 'Respondent Race',
    'T_cc': 'Race Concordance',
    'T_lay': 'Messenger Appearance',
    'T_ackn': 'Acknowledgement'
})

print("Subset shape:", q3_df.shape)

for col in ["Respondent Race", "Race Concordance", "Messenger Appearance", "Acknowledgement"]:
    print(f"\n{col}")
    print(q3_df[col].value_counts(dropna=False).sort_index())

Subset shape: (3245, 4)

Respondent Race
Respondent Race
0.0    1307
1.0    1938
Name: count, dtype: int64

Race Concordance
Race Concordance
0.0    1136
1.0    1128
NaN     981
Name: count, dtype: int64

Messenger Appearance
Messenger Appearance
0.0     479
1.0     496
NaN    2270
Name: count, dtype: int64

Acknowledgement
Acknowledgement
0.0     478
1.0     485
NaN    2282
Name: count, dtype: int64


Now go through the variables to name the ones with just 0 or 1 as responses.

In [ ]:
clean_q3_df = q3_df.replace({
    "Respondent Race":      {0: "Not Black", 1: "Black"},
    "Race Concordance":     {0: "Different Race", 1: "Same Race"},
    "Messenger Appearance": {0: "Expert", 1: "Layman"},
    "Acknowledgement":      {0: "No", 1: "Yes"},
})

In [ ]:
output_file = "/content/drive/My Drive/Colab Notebooks/q3_cleaned.csv"
clean_q3_df.to_csv(output_file, index=False)

print("Saved cleaned file to:")
print(output_file)

Saved cleaned file to:
/content/drive/My Drive/Colab Notebooks/q3_cleaned.csv


# QUESTION 4

In [ ]:
data_folder = "/content/drive/My Drive/Colab Notebooks/replication_package/2input_data/csv/"
main_survey_path = data_folder + "main_survey.csv"

if not os.path.exists(main_survey_path):
    raise FileNotFoundError(
        "main_survey.csv was not found. Run the class DTA-to-CSV conversion notebook first."
    )

raw_df = pd.read_csv(main_survey_path)

# The class conversion notebook saves the pandas index into the CSV.
# It is not part of the survey data, so remove it if it appears.
if "Unnamed: 0" in raw_df.columns:
    raw_df = raw_df.drop(columns="Unnamed: 0")

print("Raw shape:", raw_df.shape)
display(raw_df.head())
raw_df.info()


Raw shape: (2893, 164)


,id,authority,video_length,video_race,el_status,el_progress,el_durationinseconds,el_finished,el_responseid,el_flushot,...,area_division,region,region_num,region_midwest,region_northeast,region_south,region_west,missing_region,resp_no_med,resp_any_med
0,18197,doc,long,white,NaN,NaN,NaN,NaN,R_6LsRVVTs2vjptgl,NaN,...,West South Central,South,South,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1,5465,doc,short,white,NaN,NaN,NaN,NaN,R_2P0vAZKzXespjdQ,NaN,...,West South Central,South,South,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2,7534,doc,short,white,NaN,NaN,NaN,NaN,R_10ryZI0l77UTzwy,NaN,...,South Atlantic,South,South,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,5456,lay,short,black,NaN,NaN,NaN,NaN,R_2tL0vrhh2xYKLKL,NaN,...,West North Central,Midwest,Midwest,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,4758,doc,long,white,NaN,NaN,NaN,NaN,R_2v8J0lMeEAma2rt,NaN,...,Middle Atlantic,Northeast,Northeast,0.0,1.0,0.0,0.0,0.0,1.0,0.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2893 entries, 0 to 2892
Columns: 164 entries, id to resp_any_med
dtypes: float64(67), int64(69), object(28)
memory usage: 3.6+ MB


In [ ]:
data_folder = "/content/drive/My Drive/Colab Notebooks/replication_package/2input_data/csv/"
redemption_path = data_folder + "redemption_data.csv"

if not os.path.exists(redemption_path):
    raise FileNotFoundError(
        "redemption_data.csv was not found. Run the class DTA-to-CSV conversion notebook first."
    )

raw_redemption_df = pd.read_csv(redemption_path)

# The class conversion notebook saves the pandas index into the CSV.
# It is not part of the survey data, so remove it if it appears.
if "Unnamed: 0" in raw_redemption_df.columns:
    raw_redemption_df = raw_redemption_df.drop(columns="Unnamed: 0")

raw_redemption_df['redeemed'] = 'yes'

print("Raw shape:", raw_redemption_df.shape)
display(raw_redemption_df.head())
raw_redemption_df.info()

Raw shape: (30, 5)


,dateprocessed,datefilled,pharmacyname,id,redeemed
0,2020-01-03,2020-01-03,SMALL INDEPENDENT,4227.0,yes
1,2020-12-18,2020-12-18,WALGREENS,17927.0,yes
2,2020-10-29,2020-10-29,PUBLIX PHARMACY,8476.0,yes
3,2020-12-30,2020-12-30,CVS PHARMACY,17069.0,yes
4,2020-01-23,2020-01-23,RITE AID PHARMACY,5900.0,yes


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   dateprocessed  30 non-null     object 
 1   datefilled     30 non-null     object 
 2   pharmacyname   30 non-null     object 
 3   id             30 non-null     float64
 4   redeemed       30 non-null     object 
dtypes: float64(1), object(4)
memory usage: 1.3+ KB


In [ ]:
q4_df = raw_df[
    [ 'id', 'el_finished', 'likelyshot_post', 'likelyshot_pre', 'el_flushot']
].copy()

q4_df = q4_df.merge(
    raw_redemption_df[["id", "redeemed"]],
    on="id",
    how="left"
)

print(q4_df)

         id  el_finished  likelyshot_post  likelyshot_pre  el_flushot redeemed
0     18197          NaN                1               0         NaN      NaN
1      5465          NaN                0               0         NaN      NaN
2      7534          NaN                0               0         NaN      NaN
3      5456          NaN                1               2         NaN      NaN
4      4758          NaN                4               2         NaN      NaN
...     ...          ...              ...             ...         ...      ...
2888   6300          NaN               10               5         NaN      NaN
2889   8208          NaN                9               9         NaN      NaN
2890   8884          NaN                0               0         NaN      NaN
2891  14933          NaN               10              10         NaN      NaN
2892   9950          NaN                4               4         NaN      NaN

[2893 rows x 6 columns]


In [ ]:
q4_df = q4_df.rename(columns={
    'el_finished': 'completed_follow_up',
    'likelyshot_post': 'post_video_intent',
    'likelyshot_pre': 'pre_video_intent',
    'el_flushot': 'got_vaccine',
})

clean_q4_df = q4_df.replace({
    "completed_follow_up":      {0: "No", 1: "Yes"},
    "got_vaccine":      {1: "Yes", 2: "No"},
})

print(q4_df)

         id  completed_follow_up  post_video_intent  pre_video_intent  \
0     18197                  NaN                  1                 0   
1      5465                  NaN                  0                 0   
2      7534                  NaN                  0                 0   
3      5456                  NaN                  1                 2   
4      4758                  NaN                  4                 2   
...     ...                  ...                ...               ...   
2888   6300                  NaN                 10                 5   
2889   8208                  NaN                  9                 9   
2890   8884                  NaN                  0                 0   
2891  14933                  NaN                 10                10   
2892   9950                  NaN                  4                 4   

      got_vaccine redeemed  
0             NaN      NaN  
1             NaN      NaN  
2             NaN      NaN  
3      

In [ ]:
output_file = "/content/drive/My Drive/Colab Notebooks/q4_cleaned.csv"
clean_q4_df.to_csv(output_file, index=False)

print("Saved cleaned file to:")
print(output_file)

Saved cleaned file to:
/content/drive/My Drive/Colab Notebooks/q4_cleaned.csv


# Cleaning Decisions

I decided to keep NaN values as those who do not have some values are still worth considering for this project.

Main goal was to simplify format and eliminate unnecessary data entries.